In [1]:
import re

import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5645 entries, 0 to 5644
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date_start  5645 non-null   object
 1   date_end    603 non-null    object
 2   event       5645 non-null   object
dtypes: object(3)
memory usage: 132.4+ KB


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

ru_stopwords = stopwords.words("russian")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


texts = df["event"].dropna().astype(str).apply(clean_text)

vectorizer = TfidfVectorizer(
    max_df=0.9,
    min_df=5,
    stop_words=ru_stopwords,
    ngram_range=(1, 3),
    sublinear_tf=True,
)
X = vectorizer.fit_transform(texts)

n_topics = 10
model = NMF(n_components=n_topics, random_state=42)
W = model.fit_transform(X)
H = model.components_

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic in enumerate(H):
    top_words = [feature_names[j] for j in topic.argsort()[:-11:-1]]
    topics[f"Topic {i + 1}"] = top_words

print(len(vectorizer.vocabulary_))
topics

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ruslan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


3311


{'Topic 1': ['парламентские выборы',
  'парламентские',
  'выборы',
  'досрочные парламентские',
  'досрочные парламентские выборы',
  'досрочные',
  'партия',
  'большинство',
  'сирии',
  'мест'],
 'Topic 2': ['человек',
  'погибли',
  'погибли человек',
  'результате',
  'человека',
  'человек погибли',
  'получили',
  'ранены',
  'ранения',
  'погибло'],
 'Topic 3': ['должность',
  'вступил',
  'должность президента',
  'президента',
  'вступил должность',
  'вступил должность президента',
  'года',
  'президент',
  'должность президент',
  'вступил должность президент'],
 'Topic 4': ['тур',
  'выборов',
  'президентских',
  'президентских выборов',
  'второй',
  'второй тур',
  'тур президентских',
  'тур президентских выборов',
  'второй тур президентских',
  'одержал'],
 'Topic 5': ['мира',
  'чемпионат',
  'чемпионат мира',
  'россия',
  'мира хоккею',
  'хоккею',
  'чемпионат мира хоккею',
  'сборная',
  'мира хоккею шайбой',
  'хоккею шайбой'],
 'Topic 6': ['премьер',
  'мини

In [5]:
from sklearn.decomposition import TruncatedSVD

n_topics = 10

lsa = TruncatedSVD(
    n_components=n_topics,
    random_state=42
)

X_lsa = lsa.fit_transform(X)
feature_names = vectorizer.get_feature_names_out()

topics = {}

for i, comp in enumerate(lsa.components_):
    indices = np.argsort(np.abs(comp))[-12:]
    top_words = [feature_names[j] for j in indices]
    topics[f"Topic {i + 1}"] = top_words

topics

{'Topic 1': ['тур',
  'победу одержала',
  'победу одержал',
  'одержал',
  'одержала',
  'партия',
  'президентские выборы',
  'президентские',
  'победу',
  'парламентские выборы',
  'парламентские',
  'выборы'],
 'Topic 2': ['президента',
  'погибло',
  'получили ранения',
  'ранены',
  'ранения',
  'получили',
  'человек погибли',
  'человека',
  'результате',
  'погибли человек',
  'погибли',
  'человек'],
 'Topic 3': ['одержал',
  'погибли',
  'человек',
  'тур',
  'выборов',
  'парламентские',
  'парламентские выборы',
  'вступил должность',
  'должность президента',
  'вступил',
  'президента',
  'должность'],
 'Topic 4': ['одержал',
  'победу',
  'второй',
  'президентских выборов',
  'второй тур',
  'президентских',
  'вступил должность',
  'должность президента',
  'выборов',
  'тур',
  'вступил',
  'должность'],
 'Topic 5': ['хоккею шайбой',
  'мира хоккею шайбой',
  'шайбой',
  'одержала',
  'чемпионат мира хоккею',
  'хоккею',
  'мира хоккею',
  'сборная',
  'россия',
  '

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=30,
    doc_topic_prior=0.1,  # alpha
    topic_word_prior=0.01  # beta
)
X_lda = lda.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic_dist in enumerate(lda.components_):
    top_idx = topic_dist.argsort()[-12:][::-1]
    topics[f"Topic {i + 1}"] = [feature_names[j] for j in top_idx]

topics

{'Topic 1': ['стал',
  'министром',
  'премьер министром',
  'премьер',
  'новым',
  'референдум',
  'сша',
  'президентом',
  'новым премьер',
  'новым премьер министром',
  'великобритании',
  'конституционный'],
 'Topic 2': ['космический',
  'союз',
  'россии',
  'рф',
  'союз тма',
  'тма',
  'корабль',
  'аппарат',
  'посадки',
  'сша',
  'казахстане',
  'представителей'],
 'Topic 3': ['россия',
  'впервые',
  'сша',
  'европы',
  'станции',
  'истории',
  'второй тур выборов',
  'тур выборов',
  'старт',
  'выборов президента',
  'полёт',
  'оон'],
 'Topic 4': ['человек',
  'погибли',
  'результате',
  'погибли человек',
  'человека',
  'погибло',
  'ранены',
  'получили',
  'человек погибли',
  'ранения',
  'взрыв',
  'получили ранения'],
 'Topic 5': ['парламентские выборы',
  'парламентские',
  'выборы',
  'премьер',
  'президент',
  'министра',
  'премьер министра',
  'года',
  'должность',
  'министр',
  'вступил',
  'отставку'],
 'Topic 6': ['сша',
  'корабля',
  'россии',
 

In [7]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

texts = df["event"]

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    min_df=5
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts)

2026-03-01 21:10:37,945 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-03-01 21:11:21,320 - BERTopic - Embedding - Completed ✓
2026-03-01 21:11:21,321 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-01 21:11:37,708 - BERTopic - Dimensionality - Completed ✓
2026-03-01 21:11:37,709 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-01 21:11:39,358 - BERTopic - Cluster - Completed ✓
2026-03-01 21:11:39,362 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-01 21:11:39,653 - BERTopic - Representation - Completed ✓


In [8]:
import random
from collections import defaultdict


def sample_docs_per_topic(texts, topics, n_samples=10, seed=42):
    random.seed(seed)

    topic_to_docs = defaultdict(list)
    for text, topic in zip(texts, topics):
        topic_to_docs[topic].append(text)

    for topic_id, docs in sorted(topic_to_docs.items()):
        if topic_id == -1:
            print(f"topic {topic_id} | total docs: {len(docs)}")
            continue

        print("=" * 80)
        print(f"TOPIC {topic_id} | total docs: {len(docs)}")
        print("=" * 80)

        sampled = random.sample(docs, min(n_samples, len(docs)))
        for i, doc in enumerate(sampled, 1):
            print(f"{i}. {doc}")
        print()


def save_topics_barchart(topic_model: BERTopic, out_html="topics_barchart.html", top_n_topics=30):
    """
    Сохраняет интерактивный bar chart с размерами/словами тем (BERTopic).
    """
    fig = topic_model.visualize_barchart(top_n_topics=top_n_topics, n_words=10)
    fig.write_html(out_html)
    return fig


def save_documents_scatter(topic_model, texts, topics, out_html="documents_scatter.html"):
    """
    Сохраняет интерактивный scatter документов по темам (BERTopic).
    """
    fig = topic_model.visualize_documents(docs=texts, topics=topics)
    fig.write_html(out_html)
    return fig

In [9]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1394,-1_чемпионат_по_мира по_чемпионат мира по,"[чемпионат, по, мира по, чемпионат мира по, че...",[чемпионат мира по футболу в Катаре. Победу од...
1,0,180,0_перу_всеобщие_чили_президента,"[перу, всеобщие, чили, президента, всеобщие вы...",[второй тур президентских выборов в Аргентине....
2,1,146,1_самолёт_борту_на борту_все,"[самолёт, борту, на борту, все, разбился, ката...","[самолёт Ан-148, выполнявший рейс из Москвы в ..."
3,2,135,2_протеста_против_массовые_протесты,"[протеста, против, массовые, протесты, началис...",[В Турции начались массовые протесты с требова...
4,3,125,3_казахстана_казахстане_кыргызстана_армении,"[казахстана, казахстане, кыргызстана, армении,...",[Президентские выборы в Казахстане. Победил де...
5,4,104,4_результате взрыва_взрыва_результате_погибли,"[результате взрыва, взрыва, результате, погибл...",[В результате взрыва на химическом заводе во ф...
6,5,102,5_сербии_хорватии_словакии_тур,"[сербии, хорватии, словакии, тур, выборах, пар...",[Президентские и парламентские выборы в Сербии...
7,6,100,6_союз_корабля_космического корабля_космического,"[союз, корабля, космического корабля, космичес...",[приземление космического корабля Союз ТМА-01М...
8,7,86,7_сирии_ирака_ираке_аль,"[сирии, ирака, ираке, аль, войска, коалиции, б...","[Парламентские выборы в Сирии., Парламентские ..."
9,8,84,8_саммит_государств_конференция_глав,"[саммит, государств, конференция, глав, нато, ...","[саммит АТЭС (Манила, Филиппины)., саммит ШОС ..."


In [10]:
sample_docs_per_topic(texts, topics, n_samples=10)
save_topics_barchart(topic_model, out_html="../reports/topic_plots/bertopic_events_auto_barchart.html")
save_documents_scatter(topic_model, texts, topics, out_html="../reports/topic_plots/bertopic_events_auto_scatter.html")

topic -1 | total docs: 1394
TOPIC 0 | total docs: 180
1. всеобщие выборы в Мексике. Клаудия Шейнбаум была избрана первой женщиной-президентом Мексики.
2. Абель Пачеко вступил в должность президента Коста-Рики (до 8 мая 2006 года).
3. Альберто Фухимори переизбран президентом Перу. Оппозиция не признала выборы и ответила новыми маршами протеста, в которых участвовали сотни тысяч людей. Полиция жестоко подавляла выступления. В ходе столкновений сотни людей были ранены и арестованы.
4. Выборы президента Венесуэлы. Победу одержал Николас Мадуро, однако сторонники оппозиционного кандидата Энрике Каприлеса не признали результаты выборов.
5. Президентские выборы в Гватемале. По предварительным данным, победу одержал Отто Перес Молина.
6. VI-й съезд Коммунистической партии Кубы принял решение о серьёзных политических и экономических реформах в стране.
7. В Бразилии состоялись президентские выборы. Победу одержал Луис Инасиу Лула да Силва (61.3 % голосов избирателей; вступил в должность 1 января

In [11]:
# semi-supervised learning
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
    zeroshot_topic_list=[
        'природная катастрофа', 'авиакатастрофа', 'государственный переворот', 'вооруженный конфликт', 'теракт', 'протесты', 'санкции', 'спорт', 'закон'
    ],
    seed_topic_list=[
        ["землетрясение", "цунами", "извержение вулкана", "ураган", "тайфун",
         "наводнение", "оползень", "сель", "засуха", "лесной пожар"],
        ["авиакатастрофа", "крушение самолета", "пассажирский самолет",
         "на борту", "рейс", "экипаж"],
        ["государственный переворот", "госпереворот", "военный переворот",
 "свержение власти", "захват власти", "путч"],
        ["вооруженный конфликт", "война", "военные действия", "наступление", "обстрел", "ВС РФ", "ВСУ"],
        ["теракт", "смертник", "террористический акт"],
        ["санкции", "пакет санкций"],
        ["акция протеста", "протест", "массовые протесты", "беспорядки"],
        ["запуск ракеты", "космос", "спутник", "космический аппарат", "орбита", "космодром"],
        ["чемпионат мира", "спорт", "золотая медаль", "олимпийские игры", "сборная"],
        ["выборы президента", "выборы премьер-министра", "парламентские выборы"],
        ["закон", "подписание закона", "вступление в силу закона", "законопроект", "принятие закона"],
        ["Nvidia", "Microsoft", "Google", "Samsung", "Huawei", "Facebook", "Apple"]
    ]
)

topics, probs = topic_model.fit_transform(texts)

2026-03-01 21:12:25,960 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-03-01 21:13:07,979 - BERTopic - Embedding - Completed ✓
2026-03-01 21:13:07,980 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-01 21:13:08,113 - BERTopic - Guided - Completed ✓
2026-03-01 21:13:08,114 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-01 21:13:10,606 - BERTopic - Dimensionality - Completed ✓
2026-03-01 21:13:10,607 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-03-01 21:13:10,652 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-03-01 21:13:17,951 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-01 21:13:18,918 - BERTopic - Cluster - Completed ✓
2026-03-01 21:13:18,919 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2026-03-01 21:13:18,931 - BERTopic - Zeroshot Step 2 - Completed ✓
2026-03-01 21:13:18,933 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-01 21:13:19,123 - BERTopic - Representation - Completed ✓


In [12]:
topic_model.reduce_topics(
    texts,
    nr_topics=15
)

topics, probs = topic_model.transform(texts)
topic_model.get_topic_info()

2026-03-01 21:13:19,370 - BERTopic - Topic reduction - Reducing number of topics
2026-03-01 21:13:19,562 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-01 21:13:19,800 - BERTopic - Representation - Completed ✓
2026-03-01 21:13:19,803 - BERTopic - Topic reduction - Reduced number of topics from 91 to 15


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2026-03-01 21:14:02,506 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1290,-1_по_выборы_победу_на,"[по, выборы, победу, на, россии, премьер, през...",[второй тур выборов президента Словении. Побед...
1,0,1084,0_человек_погибли_результате_на,"[человек, погибли, результате, на, более, поги...",[циклон Батсирай унёс жизни в общей сложности ...
2,1,898,1_выборы_победу_выборов_тур,"[выборы, победу, выборов, тур, президента, вто...",[второй тур выборов президента Польши. Победу ...
3,2,773,2_союз_погибли_на_человек,"[союз, погибли, на, человек, сша, станции, чел...",[в Саваре (Бангладеш) обрушилось 8-этажное зда...
4,3,459,3_россии_война_вторая_рф,"[россии, война, вторая, рф, на, федерации, ден...",[Вторая чеченская война: вертолёт Ми-8МТ МВД Р...
5,4,387,4_мира_по_саммит_международный,"[мира, по, саммит, международный, россия, евро...",[чемпионат мира по современному пятиборью (Тай...
6,5,260,5_отставку_премьер_президент_союза,"[отставку, премьер, президент, союза, югослави...",[президент Йемена Абд Раббо Мансур Хади подал ...
7,6,135,6_закон_принятие_против_силу,"[закон, принятие, против, силу, сша, президент...",[Принятие шестого пакета санкций против России...
8,7,116,7_отношения_израиль_израиля_между,"[отношения, израиль, израиля, между, оон, сша,...",[Узбекистан и Парагвай установили дипломатичес...
9,8,113,8_компания_выход_системы_сша,"[компания, выход, системы, сша, китая, мире, н...",[Обанкротилась американская компания-разработч...


In [13]:
sample_docs_per_topic(texts, topics, n_samples=10)

topic -1 | total docs: 247
TOPIC 0 | total docs: 875
1. на западе Турции произошло землетрясение магнитудой от 6,6 до 6,9. Эпицентр находился в Эгейском море в 17 км от района Сеферихисар в Измире. Число погибших превысило 100 человек.
2. 24 или 25 марта — на одном из крупнейших сирийских заводов по производству боеприпасов в городе Хомс произошёл взрыв. 35 человек погибли.
3. Близ острова Сулавеси потерпел крушение индонезийский паром «Кахая Бахари» (англ. «Cahaya Bahari»); погибло 492 человека;
4. убийство подростка арабского происхождения в Нантере спровоцировало массовые беспорядки.
5. Багдадский авиаудар.
6. Вооружённый мятеж в Узбекистане: захвачена тюрьма города Андижан, на свободе оказались более 4000 заключённых.
7. Иракская война: ВВС США нанесли авиаудар по Эль-Фаллудже, в результате были жертвы среди мирного населения.
8. Вторая чеченская война: в Шатое перед зданием комендатуры подорвался на фугасе ГАЗ-66 с военнослужащими: 10 человек погибли, 7 ранено.
9. Гильдия сценарис

In [14]:
topic_model.save('../models/bertopic_events_15topics')
save_topics_barchart(topic_model, out_html="../reports/topic_plots/bertopic_events_15topics_barchart.html")
save_documents_scatter(topic_model, texts, topics, out_html="../reports/topic_plots/bertopic_events_15topics_scatter.html")

2026-03-01 21:14:02,831 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
